# 03_modeling.ipynb

**project:** Predictive Maintenance - Engine Failure Prediction
**Author:** Leart Ajro
**Purpose:** This notebook will:
             - Handle split dependent preprocessing and model training.
             - Split the preprocessed data into training and testing sets
             - Scale features using StandardScaler
             - Train and evaluate multiple models(Logistic Regression, Random Forest)
             - Compare Class imbalance handling approaches
             - Optimize for recall to minimize false negatives 

In [25]:
# Import packages 

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier

In [26]:
# Load X and y csv from notebook 2

X_resampled = pd.read_csv('../data/X_resampled.csv')
y_resampled = pd.read_csv('../data/y_resampled.csv')

In [27]:
# Confirming data loaded correctly

print(X_resampled.shape)
print(y_resampled.shape)

(19322, 5)
(19322, 1)


## Train/Test split 

Now that the data has been cleaned up and preprocessed, we will now be splitting the data  into training and testing sets to further prepare it for modeling.

This ensures the model is evaluated on unseen data and helps assess its ability to generalize to new machine conditions. Preventing the model from just memorizing the training data or also known as **overfitting**.

The testing set acts as a simulation of real world unseen data, giving us an honest measure of model performance.


In [28]:
# Splitting data into training/testing sets

X_train, X_test, y_train, y_test = train_test_split(
X_resampled,
y_resampled,
test_size=0.2,
random_state=42,
stratify=y_resampled
)

## Feature Scaling

After splitting the data into two sets we will now scale it as the features are all on vastly differing scales.

Scaling using StandardScaler transforms the data so that each feature has a mean of 0 and standard deviation of 1. Ensuring that no single feature dominated the model due to its scale.

In [29]:
# Scale using StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Model Training (Baseline) 

Now that the data has been properly processed, split, and scaled, It is time to begin training the model.

To establish a baseline for predictive performance, a Logistic Regression model is trained using the processed feature set. This provides a simple and interpretable benchmark before exploring more complex models.

In [30]:
# Logistic regression model training

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

/Users/leartajro/main/ML_Projects/predictive-maintenance/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [31]:
# Logistic regression output

print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

[[1607  326]
 [ 360 1572]]
              precision    recall  f1-score   support

           0       0.82      0.83      0.82      1933
           1       0.83      0.81      0.82      1932

    accuracy                           0.82      3865
   macro avg       0.82      0.82      0.82      3865
weighted avg       0.82      0.82      0.82      3865



## Logistic Regression (Findings)

Based on the results of the confusion matrix we can infer that running Logistic Regression was a success providing good metrics all across the board.

However with 360 failures that were not caught, that is still quite a bit and very costly in a real world enviorment. We will now be proceeding with a Random Forest model to see if we can further optimize recall.

## Random Forest

To further improve predictive performance beyond the baseline Logistical Regression model, a Random forest model will be used.

Random Forest is an ensemble method that combines multiple decision trees trained on different subsets of the data. This helps capture non linear relationships and feature interactions that a linear model may miss.

It is expected to improve the model's ability to detect machine failures by reducing bias and variance, especially in complex sensor based patterns.

In [32]:
# Random forest model training

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_scaled,y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

/Users/leartajro/main/ML_Projects/predictive-maintenance/.venv/lib/python3.14/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [33]:
# Random forest output

print(confusion_matrix(y_test,y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[[1862   71]
 [  31 1901]]
              precision    recall  f1-score   support

           0       0.98      0.96      0.97      1933
           1       0.96      0.98      0.97      1932

    accuracy                           0.97      3865
   macro avg       0.97      0.97      0.97      3865
weighted avg       0.97      0.97      0.97      3865



# Random Forest (Findings)

Looking at the results from the confusion matrix after training Random Forest we can see a massive improvement all across the board. False nrgatives went from 326 to 31 which is a drastic improvement. This improvement is attributed to Random Forest's ability to capture non-linear relationships between sensor features that Logistic regression could not.

The classification report shows a 0.97 average across the board in precision, recall, and f1 score making this model highly accurate as well.

Overall this will benefit a real world environment greatly by reducing both the time and money spent dealing with those false negatives. Even with Random Forest being a success. We will continue to try to optimize this model even further.

## Random Forest + Class Weights

Now that we have used both Logistic Regression for a baseline approach and then Random Forest as a secondary measure, which showed great improvement. I will be trying a different method of handling the class imbalance in class weights rather than using SMOTE.

Since Random Forest proved to be the strongest model, further improvememt will come from adjusting how we handle class imbalance rather than switching models entirely. Using class_weights = 'balanced' instead of SMOTE allows us to compare and contrast the two different approaches.

In [11]:
# Load original unbalanced data

df = pd.read_csv('../data/ai4i2020.csv')

In [12]:
# Drop unnecessary columns

df.drop(['UDI', 'Product ID', 'Type', 'TWF',
          'HDF', 'OSF', 'PWF', 'RNF'],axis=1, inplace=True)

In [13]:
# Redefine X and y

target = 'Machine failure'

features = ['Air temperature [K]', 
            'Process temperature [K]',
            'Rotational speed [rpm]',
            'Torque [Nm]',
            'Tool wear [min]',
           ]

X_cw = df[features] 
y_cw = df[target] 

In [14]:
# Train/test split

X_train_cw, X_test_cw, y_train_cw, y_test_cw = train_test_split(
X_cw,
y_cw,
test_size=0.2,
random_state=42,
stratify=y_cw
)

In [15]:
# Scale using StandardScaler

scaler = StandardScaler()
X_train_scaled_cw = scaler.fit_transform(X_train_cw)
X_test_scaled_cw = scaler.transform(X_test_cw)

In [18]:
# Random forest model training + class weight

rf_model_cw = RandomForestClassifier(class_weight = 'balanced', random_state=42)
rf_model_cw.fit(X_train_scaled_cw,y_train_cw)
y_pred_cw = rf_model_cw.predict(X_test_scaled_cw)

In [19]:
# Random forest model training + class weight (results)

print(confusion_matrix(y_test_cw, y_pred_cw))
print(classification_report(y_test_cw, y_pred_cw))

[[1927    5]
 [  30   38]]
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1932
           1       0.88      0.56      0.68        68

    accuracy                           0.98      2000
   macro avg       0.93      0.78      0.84      2000
weighted avg       0.98      0.98      0.98      2000



# Random Forest + Class Weight (Findings)

Running random forest with class weight instead of SMOTE as the balancing mechanism displayed some interesting results. 

As we can see based off the confusion matrix, for the 'non failure' class the model had nearly perfect accuracy. This however is misleading as high accuracy doesn't always equate to a good model. For classification problems containing extremely imbalanced datasets the most important metric is recall and that is where this model is flawed.

Displaying a 0.56 for recall in the failure class, out of 68 total failures it only caught 38 of them. This shows us a prime example of an accuracy paradox, and the big problem with imbalanced data, showing us just how powerful correctly handling that imbalance can be for the model.


# Conclusion 

Now after running both Logistic Regression and Random Forest models, and even using different methods for balancing this dataset we can come to a conclusion that Random Forest + SMOTE produced the best results among all of them. With a 97% accuracy across the board among all statistics on both classes. With false negatives being at only 31 and false positives at 71, accumulating for less than 3% of total predictions. This model was indeed the winner by a large margin.

Future improvements could include hyperparameter tuning of the Random Forest model, exploring additional features, or testing on real time streaming sensor data.
